# OCR SSOCR Test

This notebook tests the real `jiweibo/SSOCR` implementation on cropped scale-display images.

Reference repo: https://github.com/jiweibo/SSOCR

Flow:
1. crop the blue scale display from 2-3 images without YOLO
2. crop the inner LCD region
3. call the downloaded `ssocr.py` functions directly:
   `preprocess -> find_digits_positions -> recognize_digits_line_method`
4. save QC crop/mask/overlay images and show a result table

In [ ]:
from pathlib import Path
from typing import Dict, Optional, Tuple
import contextlib
import importlib.util
import io
import sys

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
from scipy import ndimage

PROJECT_ROOT = Path(r"C:\\Users\\HP\\Desktop\\Academic Presentation\\seed_project")
IMAGE_DIR = PROJECT_ROOT / "test_set" / "images"
SSOCR_PATH = PROJECT_ROOT / "a_pipeline_test" / "third_party" / "SSOCR" / "ssocr.py"
QC_DIR = PROJECT_ROOT / "a_pipeline_test" / "notebooks" / "ocr_ssocr_qc"

TEST_IMAGES = ["IMG_141.jpg", "IMG_142.jpg", "IMG_144.jpg"]
SAVE_QC = True

print("IMAGE_DIR:", IMAGE_DIR, "exists=", IMAGE_DIR.exists())
print("SSOCR_PATH:", SSOCR_PATH, "exists=", SSOCR_PATH.exists())
print("cv2:", cv2.__version__)
print("python:", sys.executable)

## Import jiweibo/SSOCR

The source file was downloaded from `https://raw.githubusercontent.com/jiweibo/SSOCR/master/ssocr.py`. Importing it is safe because its CLI parser only runs under `if __name__ == '__main__'`.

In [ ]:
def load_jiweibo_ssocr(ssocr_path: Path):
    if not ssocr_path.exists():
        raise FileNotFoundError(f"Missing SSOCR source file: {ssocr_path}")
    spec = importlib.util.spec_from_file_location("jiweibo_ssocr", ssocr_path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


ssocr = load_jiweibo_ssocr(SSOCR_PATH)
print("loaded:", ssocr.__file__)
print("functions:", ssocr.preprocess.__name__, ssocr.find_digits_positions.__name__, ssocr.recognize_digits_line_method.__name__)

## Crop And SSOCR Parameters

The screen crop is still a pure crop/color rule. No YOLO is used here. The SSOCR parameters below were tuned on the three test images in this notebook.

In [ ]:
# Blue display detection in the lower part of the image.
BLUE_SEARCH_X_RANGE = (0.20, 0.80)
BLUE_SEARCH_Y_START = 0.68
BLUE_MIN_PIXEL_COUNT = 200
BLUE_TARGET_ASPECT = 2.3

# Inner LCD crop inside the blue display bbox: left, top, right, bottom.
INNER_CROP_RATIO = (0.06, 0.06, 0.98, 0.84)

# Real jiweibo/SSOCR parameters.
SSOCR_THRESHOLD = 25
SSOCR_KERNEL_SIZE = (7, 7)
SSOCR_RESERVED_THRESHOLD = 12

# If SSOCR returns raw digits such as 1254, this inserts the scale's decimal point.
INFER_DECIMAL_PLACES = 2

print("INNER_CROP_RATIO:", INNER_CROP_RATIO)
print("SSOCR_THRESHOLD:", SSOCR_THRESHOLD)
print("SSOCR_KERNEL_SIZE:", SSOCR_KERNEL_SIZE)
print("SSOCR_RESERVED_THRESHOLD:", SSOCR_RESERVED_THRESHOLD)

In [ ]:
def load_rgb(image_path: Path) -> np.ndarray:
    return np.asarray(Image.open(image_path).convert("RGB"))


def find_blue_screen_bbox(rgb: np.ndarray) -> Optional[Tuple[int, int, int, int]]:
    h, w = rgb.shape[:2]
    x_base = int(w * BLUE_SEARCH_X_RANGE[0])
    x_end = int(w * BLUE_SEARCH_X_RANGE[1])
    y_base = int(h * BLUE_SEARCH_Y_START)
    lower = rgb[y_base:, x_base:x_end]

    r = lower[:, :, 0].astype(np.int16)
    g = lower[:, :, 1].astype(np.int16)
    b = lower[:, :, 2].astype(np.int16)
    blue_mask = (b > 100) & (b - r > 35) & (b - g > 10)
    labels, count = ndimage.label(blue_mask)

    best_bbox = None
    best_score = -1e18
    for label_id in range(1, count + 1):
        ys, xs = np.where(labels == label_id)
        if len(xs) < BLUE_MIN_PIXEL_COUNT:
            continue

        lx0, lx1 = int(xs.min()), int(xs.max()) + 1
        ly0, ly1 = int(ys.min()), int(ys.max()) + 1
        bw = lx1 - lx0
        bh = ly1 - ly0
        if bh <= 0:
            continue

        area = float(len(xs))
        cx = (lx0 + lx1) / 2.0
        ratio_penalty = abs((bw / bh) - BLUE_TARGET_ASPECT) * 500.0
        center_penalty = abs(cx - lower.shape[1] / 2.0) * 20.0
        score = area - ratio_penalty - center_penalty

        if score > best_score:
            best_score = score
            best_bbox = (x_base + lx0, y_base + ly0, x_base + lx1, y_base + ly1)

    return best_bbox


def crop_screen_inner(rgb: np.ndarray, screen_bbox: Tuple[int, int, int, int]) -> np.ndarray:
    x0, y0, x1, y1 = screen_bbox
    screen = rgb[y0:y1, x0:x1]
    sh, sw = screen.shape[:2]
    left, top, right, bottom = INNER_CROP_RATIO
    return screen[int(sh * top):int(sh * bottom), int(sw * left):int(sw * right)].copy()


def infer_decimal(raw_digits: str, decimal_places: int = INFER_DECIMAL_PLACES) -> str:
    if not raw_digits.isdigit() or len(raw_digits) <= decimal_places:
        return raw_digits
    return f"{int(raw_digits[:-decimal_places])}.{raw_digits[-decimal_places:]}"

## Run Real SSOCR

In [ ]:
def run_jiweibo_ssocr(screen_inner_rgb: np.ndarray) -> Dict[str, object]:
    crop_bgr = cv2.cvtColor(screen_inner_rgb, cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (7, 7), 0)

    # These are the real functions from jiweibo/SSOCR.
    mask = ssocr.preprocess(blurred, SSOCR_THRESHOLD, show=False, kernel_size=SSOCR_KERNEL_SIZE)
    digit_positions = ssocr.find_digits_positions(mask, reserved_threshold=SSOCR_RESERVED_THRESHOLD)

    overlay = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    with contextlib.redirect_stdout(io.StringIO()):
        digits = ssocr.recognize_digits_line_method(digit_positions, overlay, mask)

    raw_digits = "".join(str(item) for item in digits)
    return {
        "raw_digits": raw_digits,
        "pred_weight": infer_decimal(raw_digits),
        "mask": mask,
        "overlay": overlay,
        "digit_positions": digit_positions,
    }


def decode_one_image(image_path: Path) -> Dict[str, object]:
    rgb = load_rgb(image_path)
    screen_bbox = find_blue_screen_bbox(rgb)
    if screen_bbox is None:
        raise RuntimeError(f"Could not find blue display bbox: {image_path.name}")

    screen_inner = crop_screen_inner(rgb, screen_bbox)
    ssocr_result = run_jiweibo_ssocr(screen_inner)
    return {
        "image_name": image_path.name,
        "screen_bbox": tuple(int(v) for v in screen_bbox),
        "screen_inner": screen_inner,
        **ssocr_result,
    }


def save_qc(image_path: Path, result: Dict[str, object]) -> None:
    if not SAVE_QC:
        return
    QC_DIR.mkdir(parents=True, exist_ok=True)
    stem = image_path.stem
    cv2.imwrite(str(QC_DIR / f"{stem}_real_ssocr_crop.jpg"), cv2.cvtColor(result["screen_inner"], cv2.COLOR_RGB2BGR))
    cv2.imwrite(str(QC_DIR / f"{stem}_real_ssocr_mask.png"), result["mask"])
    cv2.imwrite(str(QC_DIR / f"{stem}_real_ssocr_overlay.jpg"), result["overlay"])


def show_debug(image_path: Path, result: Dict[str, object]) -> None:
    rgb = load_rgb(image_path)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].imshow(rgb)
    axes[0].set_title(image_path.name)
    axes[0].axis("off")
    x0, y0, x1, y1 = result["screen_bbox"]
    axes[0].add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="red", linewidth=2))

    axes[1].imshow(result["screen_inner"])
    axes[1].set_title("inner LCD crop")
    axes[1].axis("off")

    axes[2].imshow(result["mask"], cmap="gray")
    axes[2].set_title(f"SSOCR mask | raw={result['raw_digits']} | pred={result['pred_weight']}")
    axes[2].axis("off")
    for position in result["digit_positions"]:
        (bx0, by0), (bx1, by1) = position
        axes[2].add_patch(Rectangle((bx0, by0), bx1 - bx0, by1 - by0, fill=False, edgecolor="lime", linewidth=2))

    plt.tight_layout()
    plt.show()

In [ ]:
rows = []
for image_name in TEST_IMAGES:
    image_path = IMAGE_DIR / image_name
    assert image_path.exists(), f"missing image: {image_path}"

    result = decode_one_image(image_path)
    save_qc(image_path, result)
    show_debug(image_path, result)

    rows.append({
        "image_name": result["image_name"],
        "screen_bbox": result["screen_bbox"],
        "raw_digits": result["raw_digits"],
        "pred_weight": result["pred_weight"],
        "digit_count": len(result["digit_positions"]),
    })

result_df = pd.DataFrame(rows)
result_df

## Notes

- This notebook now calls the real `jiweibo/SSOCR` code, not a reimplemented seven-segment lookup.
- The blue display crop is still a simple image rule because this test intentionally avoids YOLO.
- If a new image fails, first tune `INNER_CROP_RATIO`, then `SSOCR_THRESHOLD`, `SSOCR_KERNEL_SIZE`, and `SSOCR_RESERVED_THRESHOLD`.
- QC files are written to `a_pipeline_test/notebooks/ocr_ssocr_qc`.